# Transit stops key validation sandbox

This notebook validates whether `(route_id, direction_id, stop_sequence)` uniquely maps to a single `stop_id` for one day of GTFS-RT TripUpdates snapshots stored in GCS.

## Workflow
1. Configure bucket + agency + day.
2. Load all `TripUpdates/<agency>/<YYYY-MM-DD>/*.pb` snapshots.
3. Flatten to stop-level observations.
4. Run uniqueness and pattern checks.
5. Export intermediate data to Parquet for reuse.

In [1]:
from google.cloud import storage
from google.transit import gtfs_realtime_pb2
import pandas as pd
from tqdm.auto import tqdm

# ---- Configure these values ----
bucket_name = "511_transit_data"  # e.g. "my-gcs-bucket" (do NOT include gs://)
prefix = "TripUpdates/muni/2026-06-23/"  # no leading slash
# ------------------------------

output_parquet = "tripupdates_stop_observations_muni_2026-06-23.parquet"

print(f"Using bucket: {bucket_name or '[NOT SET]'}")
print(f"Using prefix: {prefix}")

/Users/patri/miniconda3/envs/nachos/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Using bucket: 511_transit_data
Using prefix: TripUpdates/muni/2026-06-23/


In [2]:
# Optional: uncomment if running in a fresh environment
# !pip install google-cloud-storage gtfs-realtime-bindings pandas pyarrow tqdm

if not bucket_name.strip():
    raise ValueError(
        "bucket_name is empty. Set bucket_name in the config cell (no gs:// prefix)."
    )

client = storage.Client()
blobs = list(client.list_blobs(bucket_name.strip(), prefix=prefix))

print(f"Found {len(blobs):,} snapshot files")
if len(blobs) == 0:
    print("No files found for this prefix. Double-check bucket_name and prefix.")

rows = []

for blob in tqdm(blobs, desc="Parsing snapshots"):
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(blob.download_as_bytes())

    snapshot_ts = blob.name.split("/")[-1].replace(".pb", "")

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        tu = entity.trip_update
        trip = tu.trip

        route_id = trip.route_id if trip.route_id else None
        trip_id = trip.trip_id if trip.trip_id else None
        direction_id = trip.direction_id if trip.HasField("direction_id") else None

        for stu in tu.stop_time_update:
            stop_id = stu.stop_id if stu.stop_id else None
            stop_sequence = stu.stop_sequence if stu.HasField("stop_sequence") else None

            if route_id and stop_id is not None and stop_sequence is not None:
                rows.append(
                    {
                        "snapshot_ts": snapshot_ts,
                        "trip_id": trip_id,
                        "route_id": route_id,
                        "direction_id": direction_id,
                        "stop_sequence": int(stop_sequence),
                        "stop_id": stop_id,
                    }
                )

df = pd.DataFrame(rows)
print(f"Extracted {len(df):,} stop-level observations")

if not df.empty:
    df.to_parquet(output_parquet, index=False)
    print(f"Saved: {output_parquet}")

df.head()

Found 1,440 snapshot files


Parsing snapshots:   0%|          | 0/1440 [00:00<?, ?it/s]

Extracted 35,999,522 stop-level observations
Saved: tripupdates_stop_observations_muni_2026-06-23.parquet


,snapshot_ts,trip_id,route_id,direction_id,stop_sequence,stop_id
0,2026-06-23T00-00-48.701503Z,12053054_M41,1,0,30,13896
1,2026-06-23T00-00-48.701503Z,12053054_M41,1,0,31,13852
2,2026-06-23T00-00-48.701503Z,12053054_M41,1,0,32,13845
3,2026-06-23T00-00-48.701503Z,12053054_M41,1,0,33,13822
4,2026-06-23T00-00-48.701503Z,12053054_M41,1,0,34,13824


In [3]:
df['route_id'].unique()

array(['1', '1X', '2', '5', '6', '7', '8', '8AX', '8BX', '9', '9R', '12',
       '14R', '15', '18', '19', '22', '23', '24', '25', '27', '28', '28R',
       '29', '30', '31', '33', '35', '36', '37', '38', '38R', '39', '43',
       '44', '45', '48', '49', '52', '54', '55', '56', '57', '58', '66',
       '67', 'CA', 'F', 'J', 'K', 'L', 'M', 'N', 'PH', 'PM', 'T', '14',
       '5R', 'SF', 'FBUS', 'NOWL', '91', 'LOWL', 'TBUS', 'NBUS', 'KBUS',
       '90', '714', '30X'], dtype=object)

In [4]:
if df.empty:
    print("No rows found. Check bucket_name / agency / day values.")
else:
    # 1) Proposed key uniqueness: (route_id, direction_id, stop_sequence) -> stop_id
    key_violations = (
        df.groupby(["route_id", "direction_id", "stop_sequence"]) ["stop_id"]
        .nunique()
        .reset_index(name="n_stop_ids")
        .query("n_stop_ids > 1")
        .sort_values(["route_id", "direction_id", "stop_sequence"])
    )

    # 2) Missing direction rate
    missing_direction_rate = df["direction_id"].isna().mean()

    print(f"Rows: {len(df):,}")
    print(f"Unique routes: {df['route_id'].nunique():,}")
    print(f"Missing direction_id rate: {missing_direction_rate:.2%}")
    print(f"Key violations found: {len(key_violations):,}")

    key_violations.head(20)

Rows: 35,999,522
Unique routes: 69
Missing direction_id rate: 0.00%
Key violations found: 1,227


In [10]:
multi = df.groupby(["route_id", "direction_id", "stop_sequence"]) ["stop_id"].nunique().reset_index(name="n_stop_ids").query("n_stop_ids > 1")
multi.sort_values('n_stop_ids', ascending=False)

,route_id,direction_id,stop_sequence,n_stop_ids
2689,5,0,13,5
2693,5,0,17,5
2678,5,0,2,5
2679,5,0,3,5
2680,5,0,4,5
...,...,...,...,...
1261,29,0,44,2
1260,29,0,43,2
1259,29,0,42,2
1258,29,0,41,2


In [9]:
if df.empty:
    print("No rows to analyze")
else:
    # 3) Pattern consistency: how many distinct ordered stop patterns per route+direction?
    trip_patterns = (
        df.sort_values(["route_id", "direction_id", "trip_id", "stop_sequence"])
        .groupby(["route_id", "direction_id", "trip_id"], dropna=False)["stop_id"]
        .apply(tuple)
        .reset_index(name="stop_pattern")
    )

    pattern_counts = (
        trip_patterns.groupby(["route_id", "direction_id"], dropna=False)["stop_pattern"]
        .nunique()
        .reset_index(name="n_distinct_patterns")
        .sort_values("n_distinct_patterns", ascending=False)
    )

    print("Top route/direction groups with multiple patterns:")
    display(pattern_counts.head(20))

    # Candidate stop dimension based on proposed key
    stop_dim_candidate = (
        df[["route_id", "direction_id", "stop_sequence", "stop_id"]]
        .drop_duplicates()
        .sort_values(["route_id", "direction_id", "stop_sequence"])
        .reset_index(drop=True)
    )

    print(f"Candidate stop dimension rows: {len(stop_dim_candidate):,}")
    display(stop_dim_candidate.head(20))

Top route/direction groups with multiple patterns:


,route_id,direction_id,n_distinct_patterns
0,1,0,215
1,1,1,215
18,22,0,203
19,22,1,201
62,49,1,189
49,38R,0,186
61,49,0,185
50,38R,1,180
5,14,1,162
4,14,0,156


Candidate stop dimension rows: 6,659


,route_id,direction_id,stop_sequence,stop_id
0,1,0,1,14015
1,1,0,1,13892
2,1,0,2,16294
3,1,0,2,13875
4,1,0,3,16290
5,1,0,3,13896
6,1,0,4,16314
7,1,0,4,13852
8,1,0,5,16307
9,1,0,5,13845


In [ ]:
if df.empty:
    print("No rows to analyze")
else:
    # One record per trip with the maximum stop_sequence observed.
    trip_last_stop = (
        df.groupby(["route_id", "direction_id", "trip_id"], dropna=False)["stop_sequence"]
        .max()
        .reset_index(name="last_stop_sequence")
    )

    # Primary check requested: consistency by route.
    route_consistency = (
        trip_last_stop.groupby("route_id", dropna=False)["last_stop_sequence"]
        .agg(
            n_trips="size",
            min_last_stop="min",
            max_last_stop="max",
            n_distinct_last_stop="nunique",
        )
        .reset_index()
        .sort_values(["n_distinct_last_stop", "route_id"], ascending=[False, True])
    )

    inconsistent_routes = route_consistency.query("n_distinct_last_stop > 1")

    print("Route-level stop-count consistency (using max stop_sequence per trip)")
    print(f"Routes checked: {len(route_consistency):,}")
    print(f"Routes with inconsistent trip stop counts: {len(inconsistent_routes):,}")
    display(route_consistency.head(20))

    if not inconsistent_routes.empty:
        print("Sample inconsistent routes:")
        display(inconsistent_routes.head(20))

    # Optional diagnostic: split by direction.
    route_dir_consistency = (
        trip_last_stop.groupby(["route_id", "direction_id"], dropna=False)["last_stop_sequence"]
        .agg(
            n_trips="size",
            min_last_stop="min",
            max_last_stop="max",
            n_distinct_last_stop="nunique",
        )
        .reset_index()
        .sort_values(["n_distinct_last_stop", "route_id", "direction_id"], ascending=[False, True, True])
    )

    inconsistent_route_dirs = route_dir_consistency.query("n_distinct_last_stop > 1")

    print("\nRoute+direction diagnostic:")
    print(
        "Route+direction groups with inconsistent trip stop counts: "
        f"{len(inconsistent_route_dirs):,}"
    )
    display(route_dir_consistency.head(20))

    if not inconsistent_route_dirs.empty:
        print("Sample inconsistent route+direction groups:")
        display(inconsistent_route_dirs.head(20))